# Trait persistence via the Jacobian lens (v2): verbatim vs. inferred, across five characters

**Pre-registered, no directional prediction** — see `prediction.md`, including
the v2 series addendum committed before this rebuild. A separate question from
the negation-framing project in `jacobian-lens-experiment`; it does not extend
that project's thesis.

Reuses the already-validated pipeline: same `jlens` library, same
`google/gemma-3-4b-it` checkpoint and lens, same 35-90% depth band. No spider
check or walkthrough reproduction repeated -- that validation already happened
twice on this exact model/lens pairing.

**Before running:** `Runtime > Change runtime type > T4 GPU`. Gemma-3-4B-IT is
gated -- accept the license at https://huggingface.co/google/gemma-3-4b-it and
store an HF token as a Colab secret named `HF_token`.

**What this notebook does:**
1. Loads the model + lens (no re-validation).
2. Builds five characters x three conditions (direct / inferred / control),
   each sharing an identical mundane-town filler block and reintroduction.
3. Checks single-token status of all five trait words (`generous, patient,
   brave, curious, greedy`), bare and leading-space; halts/warns on any that
   fail so the trait can be swapped and re-registered, not silently used.
4. Locates checkpoints programmatically (period counting + character-offset
   name lookup), asserting 14 periods per text.
5. Reads each character's trait word's rank/logprob at every checkpoint, every
   condition, within the band -- plus an exploratory top-20 vocabulary dump.
6. Writes CSVs and plots per-character decay curves.

**Analysis reminder:** median rank is primary; compare decay *shape* (each
condition normalized to its own distance-0), never absolute heights; read the
reintroduction bump within-condition.

In [ ]:
# Install jlens pinned to a specific commit for reproducibility (not `main`,
# which is a moving target). Recorded 2026-07-13.
JLENS_COMMIT = "581d398613e5602a5af361e1c34d3a92ea82ba8e"
!pip install -q "git+https://github.com/anthropics/jacobian-lens.git@{JLENS_COMMIT}"
!pip install -q huggingface_hub

In [ ]:
# Gemma-3-4B-IT is gated -- reads a Colab secret named 'HF_token' instead of
# the interactive widget (key icon in the left sidebar -> add secret HF_token
# -> toggle notebook access on for this notebook).
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_token"))

## Load the model and lens

Identical to the other two notebooks: same checkpoint, same lens file, no
revision needed.

In [ ]:
import sys
import torch
import transformers
import jlens
import pandas as pd

jlens.configure_logging()

# Record the exact environment in the output, for reproducibility. Torch is
# Colab-managed (not pinned), so we log what actually ran rather than forcing a
# version; jlens is pinned in the install cell.
print("python      ", sys.version.split()[0])
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("jlens       ", getattr(jlens, "__version__", "n/a"), "@", JLENS_COMMIT[:12])
print("cuda        ", torch.version.cuda, "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GEMMA_MODEL_NAME = "google/gemma-3-4b-it"
GEMMA_LENS_REPO = "neuronpedia/jacobian-lens"
GEMMA_LENS_FILE = "gemma-3-4b-it/jlens/Salesforce-wikitext/gemma-3-4b-it_jacobian_lens.pt"

gemma_hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    GEMMA_MODEL_NAME, dtype=torch.bfloat16
).cuda()
gemma_tokenizer = transformers.AutoTokenizer.from_pretrained(GEMMA_MODEL_NAME)
gemma_model = jlens.from_hf(gemma_hf_model, gemma_tokenizer)

gemma_lens = jlens.JacobianLens.from_pretrained(GEMMA_LENS_REPO, filename=GEMMA_LENS_FILE)

n_layers = gemma_model.n_layers
# Same constraint discovered in the negation-framing project: the lens has no
# fitted Jacobian for the final layer.
LAYER_BAND = (0.35, 0.90)
band_layers = [
    l for l in range(n_layers - 1)
    if LAYER_BAND[0] <= l / (n_layers - 1) <= LAYER_BAND[1]
]
print(f"n_layers={n_layers}, band_layers ({len(band_layers)}): {band_layers}")

## Build the five characters and three conditions

Each character: a one-sentence opening, one of three triggers (direct /
inferred / control), then an identical mundane-town filler block and
reintroduction. Verbatim from the v2 addendum in `prediction.md`.

In [ ]:
FILLER_SENTENCES = [
    "The town's main street runs in a straight line from the north gate to the central square.",
    "A row of plane trees stands along the pavement, tall enough to shade the whole street by midsummer.",
    "The central square is paved with grey cobblestones laid in a fan pattern.",
    "A stone fountain in the middle of the square has three tiers and runs from spring until autumn.",
    "The nearest railway station sits at the far end of a long avenue past the clock tower.",
    "Trains to the regional capital leave twice in the morning and once in the early afternoon.",
    "The weather in this region turns from dry summers to grey, rain-heavy winters.",
    "A ridge of low hills marks the eastern edge of the district, covered in scrub and loose rock.",
    "The oldest building still standing is a squat administrative hall with a tiled roof.",
    "A single road bridge crosses the river at the southern edge, wide enough for one lane of traffic.",
]

# Each character: opening, then one of three triggers -- direct / inferred /
# control. Verbatim from prediction.md (v2 + v2.1 addenda). The inferred sentence
# never contains the trait word. The control is a trait-neutral, role-obligatory
# routine: episodic like the inferred arm, mundane, and chosen to avoid both the
# character's target trait and any lexical echo of the character's inferred scene
# (see prediction.md v2.1 for why a clean control is the hard part of the design).
CHARACTERS = [
    {
        "name": "Maria", "trait": "generous", "valence": "+", "pronoun": "her",
        "opening": "Maria runs the corner bakery in a small town outside Lyon.",
        "direct": "Maria is generous.",
        "inferred": "Every night before closing up, Maria brings part of the unsold bread to a shelter.",
        "control": "Every morning Maria unlocks the bakery and sorts the flour delivery by the back door.",
    },
    {
        "name": "Peter", "trait": "patient", "valence": "+", "pronoun": "his",
        "opening": "Peter runs a carpentry workshop in a small town outside Lyon.",
        "direct": "Peter is patient.",
        "inferred": "Peter can fish the lake all morning without a single bite and still call it time well spent.",
        "control": "Every morning Peter sweeps the sawdust from the workshop floor and lays out his tools.",
    },
    {
        "name": "Nadia", "trait": "brave", "valence": "+", "pronoun": "her",
        "opening": "Nadia works at the health clinic in a small town outside Lyon.",
        "direct": "Nadia is brave.",
        "inferred": "When fire broke out in the stairwell, Nadia went back inside twice to carry out the people who could not move.",
        "control": "Every morning Nadia checks the clinic's supply cupboard and signs the register at the desk.",
    },
    {
        "name": "Simon", "trait": "curious", "valence": "+", "pronoun": "his",
        "opening": "Simon works at a repair counter in a small town outside Lyon.",
        "direct": "Simon is curious.",
        "inferred": "Simon takes the back off every broken radio he finds, just to see how the parts fit together.",
        "control": "Every morning Simon raises the shutter and lines up the tickets for the day's repairs.",
    },
    {
        "name": "Otto", "trait": "greedy", "valence": "-", "pronoun": "his",
        "opening": "Otto manages a packing depot in a small town outside Lyon.",
        "direct": "Otto is greedy.",
        "inferred": "Otto counts his money twice a night and never once picks up a bill.",
        "control": "Every morning Otto unlocks the depot gates and marks the arriving crates against the delivery sheet.",
    },
]

CONDITIONS = ["direct", "inferred", "control"]

def reintro(name, pronoun):
    return f"It is Tuesday. {name} ties {pronoun} shoelaces."

def trigger_for(char, condition):
    if condition in ("direct", "inferred", "control"):
        return char[condition]
    raise ValueError(condition)

def build_full_text(char, condition):
    parts = [char["opening"], trigger_for(char, condition)] + FILLER_SENTENCES + [reintro(char["name"], char["pronoun"])]
    return " ".join(parts)

TEXTS = {}  # (name, condition) -> full cumulative text
for char in CHARACTERS:
    for cond in CONDITIONS:
        TEXTS[(char["name"], cond)] = build_full_text(char, cond)

# Eyeball one character across all three conditions.
for cond in CONDITIONS:
    print(f"=== Maria / {cond} ===")
    print(TEXTS[("Maria", cond)])
    print()
print(f"{len(TEXTS)} texts total ({len(CHARACTERS)} characters x {len(CONDITIONS)} conditions).")

## Target words: single-token check, one trait per character

Check both bare and leading-space forms of every character's trait word. Keep
single-token forms; drop the rest. A trait that survives in **neither** form
can't be measured -- swap it and re-register in `prediction.md` before running,
never substitute silently.

In [ ]:
# Single-token check for every character's trait word (bare and leading-space).
trait_targets = {}  # trait -> [(label, token_id), ...]
for char in CHARACTERS:
    trait = char["trait"]
    if trait in trait_targets:
        continue
    surviving = []
    for word, form in [(trait, trait), (trait, " " + trait)]:
        ids = gemma_tokenizer.encode(form, add_special_tokens=False)
        label = f"{word!r} (bare)" if form == word else f"{word!r} (leading-space)"
        if len(ids) == 1:
            surviving.append((label, ids[0]))
            print(f"KEEP  {label:<26} -> 1 token {ids} ({gemma_tokenizer.decode(ids)!r})")
        else:
            pieces = [gemma_tokenizer.decode([i]) for i in ids]
            print(f"DROP  {label:<26} -> {len(ids)} tokens {pieces}")
    trait_targets[trait] = surviving
    if not surviving:
        print(f"  !! {trait!r} is single-token in NEITHER form -- swap + re-register before running.")
    print()

print("Surviving target forms per trait:")
for trait, forms in trait_targets.items():
    print(f"  {trait:<9} {[l for l, _ in forms]}")

## Locate checkpoints, per character per condition

Programmatically: count sentence-ending periods (assert exactly 14 = 1 opening
+ 1 trigger + 10 filler + 2 reintroduction) and find the entity-mention
checkpoint as the *last* occurrence of that character's name, located by
character offset (robust to a subword split of the name). Halts rather than
silently misreading if the period count is off.

In [ ]:
def find_checkpoints(full_text, name):
    enc = gemma_tokenizer(full_text, return_tensors="pt", return_offsets_mapping=True)
    input_ids = enc["input_ids"][0]
    offsets = enc["offset_mapping"][0].tolist()
    seq_len = input_ids.shape[0]
    position_tokens = gemma_tokenizer.convert_ids_to_tokens(input_ids.tolist())

    period_positions = [i for i, t in enumerate(position_tokens) if t == "."]
    assert len(period_positions) == 14, (
        f"{name}: {len(period_positions)} periods found, expected 14 -- halt and "
        "inspect position_tokens by hand before trusting anything downstream."
    )

    checkpoints = {}
    checkpoints["distance_0_trigger_end"] = period_positions[1]
    for i in range(10):
        checkpoints[f"distance_{i+1}_filler_end"] = period_positions[2 + i]
    checkpoints["reintro_sentence_end"] = period_positions[13]

    # Entity mention = last occurrence of this character's name, mapped via
    # character offset to the token covering the name's first character. Robust
    # to a subword split of the name (exact token-string matching would silently
    # miss a split and crash on [-1]). Special tokens (BOS) carry a (0, 0) span
    # and are excluded naturally.
    name_char_start = full_text.lower().rfind(name.lower())
    assert name_char_start != -1, f"{name}: name not found in text."
    checkpoints["reintro_entity_mention"] = next(
        i for i, (s, e) in enumerate(offsets) if s <= name_char_start < e
    )
    return checkpoints, seq_len, position_tokens


all_checkpoints = {}
all_position_tokens = {}
for (name, cond), text in TEXTS.items():
    checkpoints, seq_len, position_tokens = find_checkpoints(text, name)
    all_checkpoints[(name, cond)] = checkpoints
    all_position_tokens[(name, cond)] = position_tokens
print(f"All {len(all_checkpoints)} texts pass the 14-period assert.")

# Spot-check: full checkpoints for one text ...
ex = ("Maria", "inferred")
print(f"\n=== full checkpoints: {ex[0]} / {ex[1]} ===")
for label, pos in all_checkpoints[ex].items():
    print(f"  {label:28s} -> position {pos:>4d}  token {all_position_tokens[ex][pos]!r}")

# ... and the entity-mention token for every text (confirm the name lookup lands
# on the right token in each).
print("\nentity-mention token, every text:")
for (name, cond), cps in all_checkpoints.items():
    pos = cps["reintro_entity_mention"]
    print(f"  {name:<6} {cond:<8} -> pos {pos:>4d}  token {all_position_tokens[(name, cond)][pos]!r}")

In [ ]:
# Provenance: dump the exact 15 texts that were fed in -- plus each text's
# checkpoint positions and the token actually found there -- so the run is
# auditable from disk, not reconstructed from memory. Written before the readout.
import json

stim_rows = []
for char in CHARACTERS:
    name, trait, val = char["name"], char["trait"], char["valence"]
    for cond in CONDITIONS:
        text = TEXTS[(name, cond)]
        cps = all_checkpoints[(name, cond)]
        toks = all_position_tokens[(name, cond)]
        checkpoint_tokens = {label: {"position": pos, "token": toks[pos]}
                             for label, pos in cps.items()}
        stim_rows.append({
            "name": name,
            "condition": cond,
            "trait": trait,
            "valence": val,
            "trigger": trigger_for(char, cond),
            "n_tokens": len(toks),
            "full_text": text,
            "checkpoints": json.dumps(checkpoint_tokens, ensure_ascii=False),
        })

stimuli_df = pd.DataFrame(stim_rows)
stimuli_df.to_csv("stimuli.csv", index=False)
print(f"Wrote {len(stimuli_df)} texts to stimuli.csv\n")
for _, r in stimuli_df.iterrows():
    print(f"=== {r['name']} / {r['condition']}  ({r['trait']}, {r['n_tokens']} tokens) ===")
    print(r["full_text"])
    print()

## Read the trait word at every checkpoint, every character and condition

For each (character, condition), read the rank/log-probability of that
character's trait word (surviving single-token forms) at every checkpoint,
within the band -- the pre-registered comparison. Also dump the raw top-20
J-lens vocabulary at each checkpoint (exploratory only; also the check for
wrong-target cases like a scene surfacing "kind" instead of "generous").

In [ ]:
trait_by_name = {c["name"]: c["trait"] for c in CHARACTERS}

rows = []
top20_rows = []

for (name, cond), full_text in TEXTS.items():
    trait = trait_by_name[name]
    targets = trait_targets[trait]
    if not targets:
        print(f"SKIP {name}/{cond}: no single-token form for trait {trait!r}")
        continue
    checkpoints = all_checkpoints[(name, cond)]
    positions = list(checkpoints.values())
    labels_by_position = {v: k for k, v in checkpoints.items()}

    jlens_logits, _, _ = gemma_lens.apply(
        gemma_model, full_text, layers=band_layers, positions=positions
    )
    for layer in band_layers:
        layer_logits = jlens_logits[layer]  # [len(positions), vocab]
        log_probs = torch.log_softmax(layer_logits.float(), dim=-1)
        for pos_idx, pos in enumerate(positions):
            checkpoint_label = labels_by_position[pos]
            logits_here = layer_logits[pos_idx]
            logprobs_here = log_probs[pos_idx]
            for target_label, token_id in targets:
                rank = int((logits_here > logits_here[token_id]).sum().item()) + 1
                rows.append({
                    "name": name, "trait": trait, "condition": cond,
                    "checkpoint": checkpoint_label, "position": pos, "layer": layer,
                    "depth_frac": layer / (n_layers - 1), "target": target_label,
                    "rank": rank, "logprob": float(logprobs_here[token_id].item()),
                })
            top20 = [gemma_tokenizer.decode([t]) for t in logits_here.topk(20).indices]
            for r, tok in enumerate(top20, start=1):
                top20_rows.append({
                    "name": name, "condition": cond, "checkpoint": checkpoint_label,
                    "position": pos, "layer": layer, "depth_frac": layer / (n_layers - 1),
                    "rank": r, "token": tok,
                })

    del jlens_logits, log_probs
    torch.cuda.empty_cache()

results_df = pd.DataFrame(rows)
top20_df = pd.DataFrame(top20_rows)
results_df.to_csv("stickiness_results.csv", index=False)
top20_df.to_csv("stickiness_top20.csv", index=False)
print(f"Wrote {len(results_df)} rows to stickiness_results.csv")
print(f"Wrote {len(top20_df)} rows to stickiness_top20.csv")

## Summary: median rank per checkpoint, per condition, per character

Median rank across the band is the **primary** statistic (best rank kept as
secondary). One table per character, `direct / inferred / control` side by side.
`distance_0..10` trace the decay; the two `reintro_*` points are separate events
(re-mentioning the entity is not the same as continued neutral filler). Compare
decay *shape* normalized to each condition's own distance-0, not absolute
heights.

In [ ]:
summary_rows = []
for keys, group in results_df.groupby(["name", "trait", "condition", "checkpoint", "target"]):
    name, trait, cond, checkpoint, target = keys
    summary_rows.append({
        "name": name, "trait": trait, "condition": cond, "checkpoint": checkpoint,
        "target": target, "median_rank": group["rank"].median(),
        "best_rank": group["rank"].min(),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv("stickiness_summary.csv", index=False)

pd.set_option("display.max_rows", None)
pd.set_option("display.width", 200)

distance_order = ["distance_0_trigger_end"] + [f"distance_{i}_filler_end" for i in range(1, 11)]
reintro_order = ["reintro_entity_mention", "reintro_sentence_end"]
ordered = distance_order + reintro_order
cond_order = ["direct", "inferred", "control"]

# Primary statistic: median rank. One pivot per character. If a trait kept both
# bare and leading-space forms, print each (the leading-space form is the one
# that occurs in running text).
for char in CHARACTERS:
    name = char["name"]
    sub = summary_df[summary_df["name"] == name]
    if sub.empty:
        print(f"(no data for {name})\n")
        continue
    for target in sorted(sub["target"].unique()):
        print(f"=== {name} / trait={char['trait']} / target={target}  (median rank, lower=stronger) ===")
        t = sub[sub["target"] == target]
        pivoted = t.pivot(index="checkpoint", columns="condition", values="median_rank").reindex(ordered)
        cols = [c for c in cond_order if c in pivoted.columns]
        print(pivoted[cols].to_string())
        print()

## Done

Files written:
- `stimuli.csv` -- provenance: the 15 exact texts fed in, with each text's checkpoint positions and tokens.
- `stickiness_results.csv` -- full raw sweep: every (character, condition, checkpoint, layer, target).
- `stickiness_top20.csv` -- exploratory top-20 vocabulary at every checkpoint (not part of the pre-registered comparison).
- `stickiness_summary.csv` -- median/best rank per (character, condition, checkpoint, target).
- `stickiness_decay_curves.png` -- per-character decay curves (direct / inferred / control), stars = reintroduction checkpoints.

Download all five and send them back. Reminders for the write-up: median rank is
primary; compare decay *shape* (each condition normalized to its own distance-0),
never absolute heights; read the reintroduction bump within-condition; and check
`stickiness_top20.csv` for wrong-target cases before interpreting a weak inferred
signal.

## Done

Files written:
- `stickiness_results.csv` -- full raw sweep: every (character, condition, checkpoint, layer, target).
- `stickiness_top20.csv` -- exploratory top-20 vocabulary at every checkpoint (not part of the pre-registered comparison).
- `stickiness_summary.csv` -- median/best rank per (character, condition, checkpoint, target).
- `stickiness_decay_curves.png` -- per-character decay curves (direct / inferred / control), stars = reintroduction checkpoints.

Download all four and send them back. Reminders for the write-up: median rank is
primary; compare decay *shape* (each condition normalized to its own distance-0),
never absolute heights; read the reintroduction bump within-condition; and check
`stickiness_top20.csv` for wrong-target cases before interpreting a weak inferred
signal.